# Capítulo 4: Álgebra Linear

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap04/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap04/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto. É o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem daqui —
# no livro isso vem do `execute-dir: project` do Quarto.
import os
import sys

_raiz = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_raiz, "_quarto.yml")):
    _pai = os.path.dirname(_raiz)
    if _pai == _raiz:
        raise RuntimeError("raiz do projeto não encontrada (procurando _quarto.yml)")
    _raiz = _pai
os.chdir(_raiz)
if _raiz not in sys.path:
    sys.path.insert(0, _raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 4 de Grus (2019).

> Existe algo mais inútil, ou menos útil, do que álgebra?
>
> — Billy Connolly

Álgebra linear é o ramo da matemática que trata de espaços vetoriais. Um capítulo curto não ensina álgebra linear — mas ela sustenta uma quantidade grande de conceitos e técnicas de ciência de dados, e ignorá-la seria pior do que tratá-la de forma incompleta.

O que este capítulo constrói é pequeno: um punhado de funções sobre listas de números. O que ele entrega é a base sobre a qual quase todo o resto do livro se apoia. A função `dot`, que cabe em duas linhas, é usada por onze dos módulos do livro-texto; a `distance`, por cinco — inclusive pelo classificador do [Capítulo 9](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/index.html), que você talvez já tenha lido.

> **❗ Importante — É aqui que a regra "sem bibliotecas" fica explicada**
>
> Este livro implementa tudo em Python puro e não usa `numpy` em lugar nenhum. Este é o capítulo onde essa decisão precisa de justificativa, porque é o único em que a biblioteca faria *exatamente* o que estamos construindo, só que melhor.
>
> E a justificativa não é nossa — é do próprio autor do livro-texto, que escreve, no meio deste capítulo, que usar listas como vetores é ótimo para exposição e péssimo para desempenho, e que em produção você deve usar NumPy.
>
> Ou seja: ninguém aqui está defendendo que se escreva `dot` à mão no trabalho. A questão é outra — quem nunca escreveu não sabe o que a chamada faz, e passa a vida chamando.

Ao final deste capítulo, você será capaz de:

- Representar dados numéricos como vetores e operar sobre eles componente a componente
- Implementar soma, subtração, multiplicação por escalar e média de vetores
- Explicar o que o produto escalar mede, e não apenas como calculá-lo
- Calcular magnitude e distância entre vetores, e reconhecer essas contas quando elas reaparecerem disfarçadas em outros capítulos
- Representar conjuntos de dados e relações binárias como matrizes, e avaliar quando essa representação compensa

## Seções

| Seção | Tópico |
|---|---|
| [4.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap04/01-vetores.html) | Vetores |
| [4.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap04/02-matrizes.html) | Matrizes |

## Vetores

> **📌 Nota**
>
> Esta seção corresponde a *Vectors*, do capítulo 4 de Grus (2019).

Abstratamente, **vetores** são objetos que podem ser somados entre si para formar novos vetores, e que podem ser multiplicados por *escalares* — números — também para formar novos vetores.

Concretamente, para os nossos fins, vetores são pontos em algum espaço de dimensão finita. Você pode não pensar nos seus dados como vetores, mas essa costuma ser uma forma útil de representar dado numérico.

Se você tem altura, peso e idade de muitas pessoas, pode tratar cada pessoa como um vetor de três dimensões, `[altura, peso, idade]`. Se você dá uma disciplina com quatro provas, pode tratar cada aluno como um vetor de quatro dimensões, `[prova1, prova2, prova3, prova4]`.

A abordagem mais simples, construindo do zero, é representar vetores como listas de números. Uma lista de três números corresponde a um vetor no espaço tridimensional, e vice-versa.

Fazemos isso com um *alias* de tipo, dizendo que um `Vector` é apenas uma lista de `float`:

In [ ]:
from typing import List

Vector = List[float]

height_weight_age = [70,   # polegadas
                     170,  # libras
                     40]   # anos

grades = [95,   # prova1
          80,   # prova2
          75,   # prova3
          62]   # prova4

height_weight_age, grades

> **🔷 Conceito**
>
> `Vector = List[float]` não cria um tipo novo. É um **apelido**: em tempo de execução, um `Vector` *é* uma lista comum, com todos os métodos de lista e nenhuma restrição adicional.
>
> O que ele muda é a legibilidade. Uma função declarada como `def dot(v: Vector, w: Vector) -> float` diz o que faz antes de você ler o corpo — e é essa mesma anotação que um verificador de tipos usa para acusar erro sem executar nada.
>
> Este é o primeiro exemplo de um hábito que atravessa o livro inteiro: **o código do Grus (2019) anota tipos em 168 das suas 259 funções.** Vale se acostumar.

### Aritmética componente a componente

Vamos precisar fazer aritmética com vetores. Como listas de Python não são vetores — e portanto não oferecem nenhuma facilidade aritmética —, temos que construir essas ferramentas nós mesmos.

Começando pela soma. Vetores somam **componente a componente**: se dois vetores `v` e `w` têm o mesmo comprimento, a soma deles é o vetor cujo primeiro elemento é `v[0] + w[0]`, cujo segundo é `v[1] + w[1]`, e assim por diante. Se não têm o mesmo comprimento, não é permitido somá-los.

Somar `[1, 2]` e `[2, 1]` resulta em `[1 + 2, 2 + 1]`, ou seja, `[3, 3]`.

Dá para implementar isso facilmente com `zip` e uma compreensão de lista:

In [ ]:
def add(v: Vector, w: Vector) -> Vector:
    """Soma os elementos correspondentes"""
    assert len(v) == len(w), "vectors must be the same length"

    return [v_i + w_i for v_i, w_i in zip(v, w)]

assert add([1, 2, 3], [4, 5, 6]) == [5, 7, 9]

add([1, 2, 3], [4, 5, 6])

> **❗ Importante — O `assert` é a documentação deste livro**
>
> Repare que a função tem duas linhas de `assert` com papéis completamente diferentes.
>
> O de dentro, `assert len(v) == len(w)`, é uma **verificação de contrato**: ele roda toda vez que a função é chamada e falha alto se alguém passar vetores de tamanhos diferentes. Sem ele, o `zip` truncaria em silêncio no menor dos dois, e você receberia um vetor mais curto sem qualquer aviso — que é bem pior do que um erro.
>
> O de fora, `assert add([1, 2, 3], [4, 5, 6]) == [5, 7, 9]`, é um **exemplo executável**. Ele mostra o que a função faz melhor do que uma frase, e não pode envelhecer: se alguém quebrar a `add`, a linha falha na hora em que o módulo é importado.
>
> O livro-texto usa esse padrão **417 vezes**. Não é enfeite — é como ele afirma o que cada função faz.

Subtrair funciona do mesmo jeito:

In [ ]:
def subtract(v: Vector, w: Vector) -> Vector:
    """Subtrai os elementos correspondentes"""
    assert len(v) == len(w), "vectors must be the same length"

    return [v_i - w_i for v_i, w_i in zip(v, w)]

assert subtract([5, 7, 9], [4, 5, 6]) == [1, 2, 3]

subtract([5, 7, 9], [4, 5, 6])

Às vezes vamos querer somar uma **lista** de vetores componente a componente — criar um novo vetor cujo primeiro elemento é a soma de todos os primeiros elementos, cujo segundo é a soma de todos os segundos, e assim por diante:

In [ ]:
def vector_sum(vectors: List[Vector]) -> Vector:
    """Soma todos os elementos correspondentes"""
    # Verifica que vectors não está vazio
    assert vectors, "no vectors provided!"

    # Verifica que os vetores têm todos o mesmo tamanho
    num_elements = len(vectors[0])
    assert all(len(v) == num_elements for v in vectors), "different sizes!"

    # o i-ésimo elemento do resultado é a soma de todo vector[i]
    return [sum(vector[i] for vector in vectors)
            for i in range(num_elements)]

assert vector_sum([[1, 2], [3, 4], [5, 6], [7, 8]]) == [16, 20]

vector_sum([[1, 2], [3, 4], [5, 6], [7, 8]])

Também vamos precisar multiplicar um vetor por um escalar, o que fazemos simplesmente multiplicando cada elemento por esse número:

In [ ]:
def scalar_multiply(c: float, v: Vector) -> Vector:
    """Multiplica cada elemento por c"""
    return [c * v_i for v_i in v]

assert scalar_multiply(2, [1, 2, 3]) == [2, 4, 6]

scalar_multiply(2, [1, 2, 3])

Com essas duas peças, calcular a **média componente a componente** de uma lista de vetores do mesmo tamanho sai de graça:

In [ ]:
def vector_mean(vectors: List[Vector]) -> Vector:
    """Calcula a média elemento a elemento"""
    n = len(vectors)
    return scalar_multiply(1/n, vector_sum(vectors))

assert vector_mean([[1, 2], [3, 4], [5, 6]]) == [3, 4]

vector_mean([[1, 2], [3, 4], [5, 6]])

> **🟩 Exemplo**
>
> `vector_mean` é o **centroide** de um conjunto de pontos — a posição média deles no espaço.
>
> Guarde essa função. Ela é o coração do algoritmo de k-means, no [Capítulo 17](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/index.html): agrupar pontos e recalcular o centro de cada grupo é, literalmente, chamar `vector_mean` repetidas vezes.

### O produto escalar

Uma ferramenta menos óbvia é o **produto escalar** (*dot product*). O produto escalar de dois vetores é a soma dos produtos componente a componente:

In [ ]:
def dot(v: Vector, w: Vector) -> float:
    """Calcula v_1 * w_1 + ... + v_n * w_n"""
    assert len(v) == len(w), "vectors must be same length"

    return sum(v_i * w_i for v_i, w_i in zip(v, w))

assert dot([1, 2, 3], [4, 5, 6]) == 32  # 1 * 4 + 2 * 5 + 3 * 6

dot([1, 2, 3], [4, 5, 6])

Duas linhas de código. É a função mais usada do livro inteiro.

Mas o que ela **mede**? Essa é a parte que importa, e é a que costuma ficar de fora quando se aprende a chamar a função pronta.

> **🔷 Conceito**
>
> Se `w` tem magnitude 1, o produto escalar `dot(v, w)` mede **o quanto o vetor `v` se estende na direção de `w`**.
>
> Por exemplo, se `w = [1, 0]`, então `dot(v, w)` é simplesmente a primeira componente de `v`. Outra forma de dizer a mesma coisa: é o comprimento do vetor que você obteria **projetando** `v` sobre `w`.

In [ ]:
# Figura: O produto escalar como projeção: `dot(v, w)` mede o quanto `v` avança na direção de `w`
import math
from matplotlib import pyplot as plt

v = [2.0, 1.0]
w = [0.5, 1.0]

# w normalizado, e a projeção de v sobre essa direção
mag_w = math.sqrt(w[0]**2 + w[1]**2)
u = [w[0] / mag_w, w[1] / mag_w]
escala = v[0] * u[0] + v[1] * u[1]          # dot(v, u)
proj = [escala * u[0], escala * u[1]]

fig, ax = plt.subplots(figsize=(6, 4))
ax.annotate("", xy=v, xytext=(0, 0),
            arrowprops=dict(arrowstyle="-|>", lw=2, color="#1f4e79"))
ax.annotate("", xy=w, xytext=(0, 0),
            arrowprops=dict(arrowstyle="-|>", lw=2, color="#1f4e79"))
ax.annotate("", xy=proj, xytext=(0, 0),
            arrowprops=dict(arrowstyle="-|>", lw=2, color="#c00000"))
ax.plot([v[0], proj[0]], [v[1], proj[1]], ls=":", color="0.4")

ax.text(v[0] + 0.06, v[1], "v", fontsize=12)
ax.text(w[0] + 0.06, w[1], "w", fontsize=12)
ax.text(proj[0] - 0.30, proj[1] + 0.10, "projeção de v em w", fontsize=9, color="#c00000")

ax.set_xlim(-0.3, 2.6)
ax.set_ylim(-0.3, 1.8)
ax.set_aspect("equal")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

Usando o produto escalar, fica fácil calcular a **soma dos quadrados** de um vetor:

In [ ]:
def sum_of_squares(v: Vector) -> float:
    """Retorna v_1 * v_1 + ... + v_n * v_n"""
    return dot(v, v)

assert sum_of_squares([1, 2, 3]) == 14  # 1 * 1 + 2 * 2 + 3 * 3

sum_of_squares([1, 2, 3])

E, com ela, a **magnitude** (ou comprimento) do vetor:

In [ ]:
import math

def magnitude(v: Vector) -> float:
    """Retorna a magnitude (ou comprimento) de v"""
    return math.sqrt(sum_of_squares(v))   # math.sqrt é a raiz quadrada

assert magnitude([3, 4]) == 5

magnitude([3, 4])

> **🟩 Exemplo**
>
> `magnitude([3, 4]) == 5` é o triângulo 3-4-5 do teorema de Pitágoras, que você viu no ensino médio.
>
> Não é coincidência: a magnitude de um vetor **é** a hipotenusa, generalizada para qualquer número de dimensões. Toda vez que este livro medir "quão longe" ou "quão grande", vai ser esta conta.

### Distância

Agora temos todas as peças para calcular a distância entre dois vetores, definida como:

$$\sqrt{(v_1 - w_1)^2 + \dots + (v_n - w_n)^2}$$

Em código:

In [ ]:
def squared_distance(v: Vector, w: Vector) -> float:
    """Calcula (v_1 - w_1) ** 2 + ... + (v_n - w_n) ** 2"""
    return sum_of_squares(subtract(v, w))

def distance(v: Vector, w: Vector) -> float:
    """Calcula a distância entre v e w"""
    return math.sqrt(squared_distance(v, w))

distance([1, 2, 3], [4, 5, 6])

Talvez fique mais claro escrita assim — é a mesma função:

In [ ]:
def distance(v: Vector, w: Vector) -> float:
    return magnitude(subtract(v, w))

distance([1, 2, 3], [4, 5, 6])

> **🔷 Conceito**
>
> Compare as duas versões. A primeira diz *como calcular*: subtraia, eleve ao quadrado, some, tire a raiz. A segunda diz *o que é*: **a distância entre dois pontos é a magnitude da diferença entre eles**.
>
> As duas produzem exatamente o mesmo número. A segunda é preferível não por ser mais curta, mas porque nomeia a ideia — e ideias com nome são as que você reconhece quando reaparecem.

Isso é o bastante para começar. Estas funções vão ser usadas intensamente pelo resto do livro.

> **💡 Dica — Na prática: NumPy — e por que não o usamos aqui**
>
> Tudo o que você construiu nesta seção existe pronto, mais rápido e mais completo:
>
> ```python
> import numpy as np
>
> v = np.array([1, 2, 3])
> w = np.array([4, 5, 6])
>
> v + w                      # add
> v - w                      # subtract
> 2 * v                      # scalar_multiply
> np.mean([v, w], axis=0)    # vector_mean
> v @ w                      # dot
> np.linalg.norm(v)          # magnitude
> np.linalg.norm(v - w)      # distance
> ```
>
> A diferença de desempenho não é pequena: o NumPy guarda os números num bloco contíguo de memória e executa as operações em código compilado e vetorizado, enquanto a nossa lista de `float` é um vetor de ponteiros para objetos Python, percorrido um por um pelo interpretador. Em vetores grandes, a diferença é de ordens de grandeza.
>
> **O autor do livro-texto diz isso explicitamente**, no meio deste mesmo capítulo: usar listas como vetores é ótimo para exposição e péssimo para desempenho, e em código de produção você deve usar o NumPy. E encerra o capítulo observando que toda essa maquinaria vem de graça com a biblioteca.
>
> Então a regra deste livro — nada de NumPy — não é uma opinião sobre engenharia. É uma escolha sobre **ordem de aprendizado**. `v @ w` devolve um número; ele não lhe diz que aquele número mede projeção, nem que ele vira zero quando os vetores são perpendiculares, nem por que ele aparece no meio de uma rede neural. Depois desta seção, você olha `v @ w` e enxerga a soma dos produtos.

## Matrizes

> **📌 Nota**
>
> Esta seção corresponde a *Matrices*, do capítulo 4 de Grus (2019).

Uma **matriz** é uma coleção bidimensional de números. Vamos representá-las como listas de listas, em que cada lista interna tem o mesmo tamanho e representa uma *linha* da matriz.

Se `A` é uma matriz, então `A[i][j]` é o elemento da *i*-ésima linha e da *j*-ésima coluna. Seguindo a convenção matemática, vamos frequentemente usar letras maiúsculas para nomear matrizes:

In [ ]:
from typing import List

Vector = List[float]

# Outro alias de tipo
Matrix = List[List[float]]

A = [[1, 2, 3],   # A tem 2 linhas e 3 colunas
     [4, 5, 6]]

B = [[1, 2],      # B tem 3 linhas e 2 colunas
     [3, 4],
     [5, 6]]

A, B

> **⚠️ Atenção**
>
> Em matemática, você normalmente chamaria a primeira linha da matriz de "linha 1" e a primeira coluna de "coluna 1".
>
> Como estamos representando matrizes com listas de Python, que são indexadas a partir de zero, vamos chamá-las de "linha 0" e "coluna 0". Essa diferença de uma unidade é uma fonte inesgotável de erro para quem traduz uma fórmula de um livro de matemática para código — vale ter consciência dela toda vez.

### Forma, linhas e colunas

Dada essa representação de lista de listas, a matriz `A` tem `len(A)` linhas e `len(A[0])` colunas, o que chamamos de sua **forma** (*shape*):

In [ ]:
from typing import Tuple

def shape(A: Matrix) -> Tuple[int, int]:
    """Retorna (# de linhas de A, # de colunas de A)"""
    num_rows = len(A)
    num_cols = len(A[0]) if A else 0   # número de elementos da primeira linha
    return num_rows, num_cols

assert shape([[1, 2, 3], [4, 5, 6]]) == (2, 3)   # 2 linhas, 3 colunas

shape(A), shape(B)

Se uma matriz tem $n$ linhas e $k$ colunas, dizemos que é uma matriz $n \times k$. Podemos pensar em cada linha de uma matriz $n \times k$ como um vetor de tamanho $k$, e em cada coluna como um vetor de tamanho $n$:

In [ ]:
def get_row(A: Matrix, i: int) -> Vector:
    """Retorna a i-ésima linha de A (como um Vector)"""
    return A[i]                # A[i] já é a i-ésima linha

def get_column(A: Matrix, j: int) -> Vector:
    """Retorna a j-ésima coluna de A (como um Vector)"""
    return [A_i[j]             # j-ésimo elemento da linha A_i
            for A_i in A]      # para cada linha A_i

get_row(A, 0), get_column(A, 1)

> **🟩 Exemplo**
>
> Repare na assimetria de custo entre as duas funções, que a representação impõe.
>
> Pegar uma linha é imediato: ela já existe como lista. Pegar uma coluna exige percorrer **todas** as linhas montando uma lista nova. Numa matriz de um milhão de linhas, `get_row` é instantânea e `get_column` percorre o milhão.
>
> Nada disso é acidente da nossa implementação — é consequência de ter escolhido guardar por linhas. Quem escolhe uma estrutura de dados escolhe também quais operações vão ser baratas.

Também vamos querer criar uma matriz a partir da sua forma e de uma função que gera os elementos. Dá para fazer isso com uma compreensão de lista aninhada:

In [ ]:
from typing import Callable

def make_matrix(num_rows: int,
                num_cols: int,
                entry_fn: Callable[[int, int], float]) -> Matrix:
    """
    Retorna uma matriz num_rows x num_cols
    cuja entrada (i, j) é entry_fn(i, j)
    """
    return [[entry_fn(i, j)             # dado i, cria uma lista
             for j in range(num_cols)]  #   [entry_fn(i, 0), ... ]
            for i in range(num_rows)]   # cria uma lista para cada i

Com essa função, dá para construir uma matriz **identidade** 5 × 5 — com 1 na diagonal e 0 no resto — assim:

In [ ]:
def identity_matrix(n: int) -> Matrix:
    """Retorna a matriz identidade n x n"""
    return make_matrix(n, n, lambda i, j: 1 if i == j else 0)

assert identity_matrix(5) == [[1, 0, 0, 0, 0],
                              [0, 1, 0, 0, 0],
                              [0, 0, 1, 0, 0],
                              [0, 0, 0, 1, 0],
                              [0, 0, 0, 0, 1]]

identity_matrix(5)

> **📌 Nota**
>
> Honestidade sobre o que acabamos de construir: `get_row`, `get_column` e `identity_matrix` **não são usadas por nenhum outro capítulo deste livro**.
>
> Elas estão aqui porque completam o quadro — são o vocabulário mínimo com que qualquer texto de álgebra linear conversa, e você vai encontrá-las em qualquer biblioteca que use depois. Mas vale saber a diferença entre o que este capítulo entrega para o resto do livro (`dot`, `distance`, `vector_mean`, `magnitude`, `make_matrix`) e o que ele entrega para a sua formação.

### Para que servem matrizes

Matrizes vão nos importar por três razões.

**Primeira: uma matriz representa um conjunto de dados formado por vários vetores**, bastando considerar cada vetor como uma linha. Se você tem altura, peso e idade de mil pessoas, pode colocá-las numa matriz de 1.000 × 3:

In [ ]:
data = [[70, 170, 40],
        [65, 120, 26],
        [77, 250, 19],
        # ....
       ]

shape(data)

Esse é o formato em que praticamente todo dado tabular chega: uma linha por observação, uma coluna por variável.

**Segunda: uma matriz $n \times k$ pode representar uma função linear** que leva vetores de $k$ dimensões em vetores de $n$ dimensões. Várias técnicas e conceitos adiante envolvem funções desse tipo — é o que uma camada de rede neural faz. O [Capítulo 15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/index.html) constrói a primeira delas, ainda guardada como uma lista de vetores de peso, um por neurônio; é o [Capítulo 16](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/index.html) que a escreve como matriz de fato, com a forma $n \times k$ desta linha, e cobra a dívida com todas as letras.

**Terceira: matrizes representam relações binárias.** No [Capítulo 1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap01/index.html), representamos as arestas de uma rede como uma coleção de pares. Uma representação alternativa é criar uma matriz `A` em que `A[i][j]` vale 1 se os nós `i` e `j` estão conectados, e 0 caso contrário.

Lá tínhamos:

In [ ]:
friendships = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 3), (3, 4),
               (4, 5), (5, 6), (5, 7), (6, 8), (7, 8), (8, 9)]

A mesma informação, como matriz:

In [ ]:
#                user 0  1  2  3  4  5  6  7  8  9
friend_matrix = [[0, 1, 1, 0, 0, 0, 0, 0, 0, 0],  # user 0
                 [1, 0, 1, 1, 0, 0, 0, 0, 0, 0],  # user 1
                 [1, 1, 0, 1, 0, 0, 0, 0, 0, 0],  # user 2
                 [0, 1, 1, 0, 1, 0, 0, 0, 0, 0],  # user 3
                 [0, 0, 0, 1, 0, 1, 0, 0, 0, 0],  # user 4
                 [0, 0, 0, 0, 1, 0, 1, 1, 0, 0],  # user 5
                 [0, 0, 0, 0, 0, 1, 0, 0, 1, 0],  # user 6
                 [0, 0, 0, 0, 0, 1, 0, 0, 1, 0],  # user 7
                 [0, 0, 0, 0, 0, 0, 1, 1, 0, 1],  # user 8
                 [0, 0, 0, 0, 0, 0, 0, 0, 1, 0]]  # user 9

shape(friend_matrix)

Se há poucas conexões, essa representação é bem mais ineficiente, porque você acaba guardando um monte de zeros. Em compensação, com a matriz fica muito mais rápido verificar se dois nós estão conectados: em vez de percorrer a lista de arestas, basta uma consulta direta.

In [ ]:
assert friend_matrix[0][2] == 1, "0 and 2 are friends"
assert friend_matrix[0][8] == 0, "0 and 8 are not friends"

friend_matrix[0][2], friend_matrix[0][8]

E, para encontrar as conexões de um nó, basta olhar a linha (ou a coluna) correspondente:

In [ ]:
# basta olhar uma linha
friends_of_five = [i
                   for i, is_friend in enumerate(friend_matrix[5])
                   if is_friend]

friends_of_five

> **🔷 Conceito**
>
> Compare os dois custos, com $n$ usuários e $m$ amizades.
>
> **Lista de pares:** ocupa espaço proporcional a $m$ — só o que existe. Mas perguntar "0 e 2 são amigos?" exige percorrer a lista inteira: $O(m)$.
>
> **Matriz de adjacência:** responde a mesma pergunta com uma consulta direta, $O(1)$. Mas ocupa $n^2$ posições, **existindo a amizade ou não**.
>
> Com dez usuários, a matriz tem 100 posições e 24 delas são diferentes de zero. Numa rede social de verdade, com milhões de usuários e algumas centenas de amigos cada, a matriz teria trilhões de posições quase todas zeradas — e não caberia em memória nenhuma.
>
> Não existe representação certa; existe a que serve à pergunta que você faz. Essa escolha reaparece em quase todo problema deste livro.

Com um grafo pequeno, dá simplesmente para guardar a lista de conexões junto de cada nó e acelerar tudo. Mas, num grafo grande e em evolução, isso provavelmente sairia caro demais para manter.

Matrizes voltam ao longo do livro inteiro.

> **💡 Dica — Na prática: NumPy e matrizes esparsas**
>
> Como na seção anterior, tudo isto existe pronto:
>
> ```python
> import numpy as np
>
> A = np.array([[1, 2, 3], [4, 5, 6]])
>
> A.shape          # shape
> A[0]             # get_row
> A[:, 1]          # get_column — e aqui SEM percorrer as linhas em Python
> np.eye(5)        # identity_matrix
> A @ B            # multiplicação de matrizes, que nem chegamos a implementar
> ```
>
> Repare em `A[:, 1]`. No nosso código, pegar uma coluna custa percorrer todas as linhas; no NumPy, o array conhece o próprio *layout* de memória e devolve uma visão da coluna sem copiar nada. É a mesma ideia com um custo diferente — e você só percebe a diferença se tiver escrito a versão cara antes.
>
> Para o caso da matriz de adjacência quase toda zerada, existe uma família inteira de estruturas em `scipy.sparse`, que guardam só as posições não nulas e recuperam boa parte da velocidade de consulta sem pagar os $n^2$ de memória. É a resposta de engenharia para o dilema que a seção acabou de descrever.

## Leituras adicionais

A seção "For Further Exploration" do capítulo 4 de Grus (2019) indica três livros de álgebra linear disponíveis gratuitamente:

- *Linear Algebra*, de Jim Hefferon (Saint Michael's College)
- *Linear Algebra*, de David Cherney, Tom Denton, Rohit Thomas e Andrew Waldron (UC Davis)
- *Linear Algebra Done Wrong*, de Sergei Treil (Brown University) — uma introdução mais avançada, para quem estiver disposto

E encerra observando que toda a maquinaria construída aqui vem pronta no [NumPy](https://numpy.org), junto de bastante coisa a mais.

## Referências

- **Grus**. *Data Science from Scratch: First Principles with Python*. 2nd ed.. O'Reilly Media. 2019.